### 라마인덱스 준비

In [1]:
# 패키지 설치
!pip install llama-index==0.10.39
!pip install llama-index-llms-gemini
!pip install llama-index-embeddings-huggingface

  Using cached llama_index_core-0.10.68.post1-py3-none-any.whl.metadata (2.5 kB)
Using cached llama_index_core-0.10.68.post1-py3-none-any.whl (1.6 MB)
  Attempting uninstall: llama-index-core
    Found existing installation: llama-index-core 0.11.19
    Uninstalling llama-index-core-0.11.19:
      Successfully uninstalled llama-index-core-0.11.19
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index-embeddings-huggingface 0.3.1 requires llama-index-core<0.12.0,>=0.11.0, but you have llama-index-core 0.10.68.post1 which is incompatible.
llama-index-llms-gemini 0.3.7 requires llama-index-core<0.12.0,>=0.11.0, but you have llama-index-core 0.10.68.post1 which is incompatible.
  Using cached llama_index_core-0.11.19-py3-none-any.whl.metadata (2.4 kB)
Using cached llama_index_core-0.11.19-py3-none-any.whl (1.6 MB)
  Attempting uninstall: llama-index-core
    F

In [2]:
# 환경 변수 준비(좌측 하단의 열쇠 아이콘으로 GOOGLE_API_KEY 설정)
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [3]:
import logging
import sys

# 로그 레벨 설정
logging.basicConfig(stream=sys.stdout, level=logging.DEBUG, force=True)

In [4]:
from llama_index.core import Settings
from llama_index.llms.gemini import Gemini
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# LLM 모델 준비
Settings.llm = Gemini(
    model_name="models/gemini-1.5-flash",
    safety_settings={
        "HARM_CATEGORY_HARASSMENT": "BLOCK_NONE",
        "HARM_CATEGORY_HATE_SPEECH": "BLOCK_NONE",
        "HARM_CATEGORY_SEXUALLY_EXPLICIT" : "BLOCK_NONE",
        "HARM_CATEGORY_DANGEROUS_CONTENT" : "BLOCK_NONE"
    }
)

# 임베딩 모델 준비
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-m3"
)

INFO:numexpr.utils:NumExpr defaulting to 2 threads.
DEBUG:asyncio:Using selector: EpollSelector
DEBUG:pydot:pydot initializing
DEBUG:pydot:pydot 3.0.2
DEBUG:pydot.dot_parser:pydot dot_parser module initializing
DEBUG:pydot.core:pydot core module initializing
DEBUG:h5py._conv:Creating converter from 7 to 5
DEBUG:h5py._conv:Creating converter from 5 to 7
DEBUG:h5py._conv:Creating converter from 7 to 5
DEBUG:h5py._conv:Creating converter from 5 to 7
DEBUG:tensorflow:Falling back to TensorFlow client; we recommended you install the Cloud TPU client directly with pip install cloud-tpu-client.
DEBUG:jax._src.path:etils.epath found. Using etils.epath for file I/O.
DEBUG:urllib3.util.retry:Converted retries value: 3 -> Retry(total=3, connect=None, read=None, redirect=None, status=None)
DEBUG:urllib3.connectionpool:Starting new HTTP connection (1): localhost:41371
INFO:tornado.access:200 GET /v1beta/models/gemini-1.5-flash?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 1360.38ms
DEBUG:urllib3.co

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /BAAI/bge-m3/resolve/main/README.md HTTP/11" 200 0
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /BAAI/bge-m3/resolve/main/modules.json HTTP/11" 200 0
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /BAAI/bge-m3/resolve/main/sentence_bert_config.json HTTP/11" 200 0
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /BAAI/bge-m3/resolve/main/config.json HTTP/11" 200 0
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /BAAI/bge-m3/resolve/main/model.safetensors HTTP/11" 404 0
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /BAAI/bge-m3/resolve/main/tokenizer_config.json HTTP/11" 200 0
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "GET /api/models/BAAI/bge-m3/revision/main HTTP/11" 200 4411
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "GET /api/models/BAAI/bge-m3 HTTP/11" 200 4411
INFO:sentence_transformers.SentenceTransformer:2 prompts are 

### 라마인덱스를 활용한 질의 응답

In [5]:
from llama_index.core import SimpleDirectoryReader

# 문서 불러오기 (data 폴더에 문서를 먼저 업로드)
documents = SimpleDirectoryReader("data").load_data()

DEBUG:llama_index.core.readers.file.base:> [SimpleDirectoryReader] Total files added: 7
DEBUG:fsspec.local:open file: /content/data/redhood1.txt
DEBUG:fsspec.local:open file: /content/data/redhood2.txt
DEBUG:fsspec.local:open file: /content/data/redhood3.txt
DEBUG:fsspec.local:open file: /content/data/redhood4.txt
DEBUG:fsspec.local:open file: /content/data/redhood5.txt
DEBUG:fsspec.local:open file: /content/data/redhood6.txt
DEBUG:fsspec.local:open file: /content/data/redhood7.txt


In [6]:
from llama_index.core import VectorStoreIndex

# 인덱스 작성
index = VectorStoreIndex.from_documents(documents)

DEBUG:llama_index.core.node_parser.node_utils:> Adding chunk: 1장: 밤의 도시, 네온 불빛 아래에서
장소: 2077년, 밤의 도시, 네온 불빛이...
DEBUG:llama_index.core.node_parser.node_utils:> Adding chunk: 2장: 잭과의 만남, 위험한 동행
장소: 은비의 은신처, 폐허가 된 건물

은비...
DEBUG:llama_index.core.node_parser.node_utils:> Adding chunk: 3장: 아크 코퍼레이션 잠입 작전
장소: 아크 코퍼레이션 본사, 최첨단 보안 시스템...
DEBUG:llama_index.core.node_parser.node_utils:> Adding chunk: 4장: 데이터의 바다에서 진실을 찾아서
장소: 아크 코퍼레이션 데이터 센터

은...
DEBUG:llama_index.core.node_parser.node_utils:> Adding chunk: 5장: 추격과 탈출
장소: 도시의 뒷골목, 고속도로

아크 코퍼레이션의 추격대가...
DEBUG:llama_index.core.node_parser.node_utils:> Adding chunk: 6장: 세상에 알리다
장소: 은비의 블로그

은비는 자신이 얻은 증거를 바탕으로...
DEBUG:llama_index.core.node_parser.node_utils:> Adding chunk: 7장: 새로운 시작
장소: 폐허가 된 건물

은비와 잭은 다시 낡은 건물의 지하...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
# 쿼리 엔진 작성
query_engine = index.as_query_engine()

In [8]:
# 질의 응답 1
print(query_engine.query("은비의 나이는?"))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

DEBUG:llama_index.core.indices.utils:> Top 2 nodes:
> [Node d2768793-9b08-4019-9e30-7a1e5308ff4d] [Similarity score:             0.536515] 1장: 밤의 도시, 네온 불빛 아래에서
장소: 2077년, 밤의 도시, 네온 불빛이 가득한 뒷골목

등장인물: 은비(17세, 해커), 잭(은비의 파트너, 전직 용병)
...
> [Node e1826662-7a6b-4891-b44d-bae99f820664] [Similarity score:             0.447299] 2장: 잭과의 만남, 위험한 동행
장소: 은비의 은신처, 폐허가 된 건물

은비는 낡은 건물의 지하실에 도착했다. 그곳에는 잭이 기다리고 있었다. 잭은 거대한 체구에 온...
DEBUG:urllib3.util.retry:Converted retries value: 3 -> Retry(total=3, connect=None, read=None, redirect=None, status=None)
DEBUG:urllib3.connectionpool:Starting new HTTP connection (1): localhost:41371
INFO:tornado.access:200 POST /v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 1535.62ms
DEBUG:urllib3.connectionpool:http://localhost:41371 "POST /v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint HTTP/11" 200 448
은비는 17세입니다. 



In [9]:
# 질의 응답 2
print(query_engine.query("은비가 은신처에서 만난 인물은?"))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

DEBUG:llama_index.core.indices.utils:> Top 2 nodes:
> [Node e1826662-7a6b-4891-b44d-bae99f820664] [Similarity score:             0.523242] 2장: 잭과의 만남, 위험한 동행
장소: 은비의 은신처, 폐허가 된 건물

은비는 낡은 건물의 지하실에 도착했다. 그곳에는 잭이 기다리고 있었다. 잭은 거대한 체구에 온...
> [Node d2768793-9b08-4019-9e30-7a1e5308ff4d] [Similarity score:             0.481006] 1장: 밤의 도시, 네온 불빛 아래에서
장소: 2077년, 밤의 도시, 네온 불빛이 가득한 뒷골목

등장인물: 은비(17세, 해커), 잭(은비의 파트너, 전직 용병)
...
INFO:tornado.access:200 POST /v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 1033.10ms
DEBUG:urllib3.connectionpool:http://localhost:41371 "POST /v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint HTTP/11" 200 529
은비는 낡은 건물의 지하실에서 잭을 만났습니다. 



In [10]:
# 질의 응답 3
print(query_engine.query("은비와 잭이 쫓는 집단의 정체는?"))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

DEBUG:llama_index.core.indices.utils:> Top 2 nodes:
> [Node 40b2c7f3-f871-43ef-894c-1b403e7417d7] [Similarity score:             0.55737] 5장: 추격과 탈출
장소: 도시의 뒷골목, 고속도로

아크 코퍼레이션의 추격대가 은비와 잭을 뒤쫓았다. 둘은 도시를 가로지르는 고속도로에서 격렬한 추격전을 벌였다. 은비...
> [Node d2768793-9b08-4019-9e30-7a1e5308ff4d] [Similarity score:             0.532909] 1장: 밤의 도시, 네온 불빛 아래에서
장소: 2077년, 밤의 도시, 네온 불빛이 가득한 뒷골목

등장인물: 은비(17세, 해커), 잭(은비의 파트너, 전직 용병)
...
INFO:tornado.access:200 POST /v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 1159.19ms
DEBUG:urllib3.connectionpool:http://localhost:41371 "POST /v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint HTTP/11" 200 554
은비와 잭은 아크 코퍼레이션의 추격대에 쫓기고 있습니다. 



### 인덱스 저장하고 불러오기

In [11]:
# 인덱스 저장
index.storage_context.persist()

DEBUG:fsspec.local:open file: /content/storage/docstore.json
DEBUG:fsspec.local:open file: /content/storage/index_store.json
DEBUG:fsspec.local:open file: /content/storage/graph_store.json
DEBUG:fsspec.local:open file: /content/storage/default__vector_store.json
DEBUG:fsspec.local:open file: /content/storage/image__vector_store.json


In [12]:
from llama_index.core import StorageContext, load_index_from_storage

# 인덱스 불러오기
storage_context = StorageContext.from_defaults(persist_dir="./storage")
index = load_index_from_storage(storage_context)

DEBUG:llama_index.core.storage.kvstore.simple_kvstore:Loading llama_index.core.storage.kvstore.simple_kvstore from ./storage/docstore.json.
DEBUG:fsspec.local:open file: /content/storage/docstore.json
DEBUG:llama_index.core.storage.kvstore.simple_kvstore:Loading llama_index.core.storage.kvstore.simple_kvstore from ./storage/index_store.json.
DEBUG:fsspec.local:open file: /content/storage/index_store.json
DEBUG:llama_index.core.graph_stores.simple:Loading llama_index.core.graph_stores.simple from ./storage/graph_store.json.
DEBUG:fsspec.local:open file: /content/storage/graph_store.json
DEBUG:fsspec.local:open file: /content/storage/property_graph_store.json
DEBUG:llama_index.core.vector_stores.simple:Loading llama_index.core.vector_stores.simple from ./storage/default__vector_store.json.
DEBUG:fsspec.local:open file: /content/storage/default__vector_store.json
DEBUG:llama_index.core.vector_stores.simple:Loading llama_index.core.vector_stores.simple from ./storage/image__vector_store.js